# Jersey Number Recognition — Pipeline
Run cells top-to-bottom. Each stage is independent — skip preprocessing if crops already exist.

In [7]:
# ── Colab setup: clone repo and install dependencies ──────────────────────
# Run this cell ONCE at the start of each Colab session.
import os, subprocess

REPO_URL    = 'https://github.com/SerenaChen7/COSC419B_2025W2.git'
BRANCH      = 'tithi/integration'
CLONE_DIR   = '/content/COSC419B_2025W2'

if not os.path.isdir(CLONE_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, CLONE_DIR], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print('Repo updated.')

# Install Python dependencies
subprocess.run(['pip', 'install', '-q', '-r', f'{CLONE_DIR}/requirements.txt'], check=True)
print('Dependencies installed.')

Repo cloned.
Dependencies installed.


In [8]:
import os
import sys
import subprocess

# ── Move to project root (needed when running in Colab) ───────────────────
if not os.path.isdir('src'):
    import glob
    # Search /content for any directory that has src/train.py inside it
    matches = glob.glob('/content/*/src/train.py')
    if matches:
        os.chdir(os.path.dirname(os.path.dirname(matches[0])))
    else:
        print('WARNING: Could not find project root. Clone the repo under /content first.')
print('Working directory:', os.getcwd())

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR         = 'data/jersey-2023'
OUTPUT_DIR       = 'outputs'
CROPS_DIR_TRAIN  = 'data/jersey-2023/train/crops'
CROPS_DIR_TEST   = 'data/jersey-2023/test/crops'
CHECKPOINT       = 'outputs/best_model.pth'
PREDICTIONS_FILE = 'outputs/predictions.json'
GT_TEST          = 'data/jersey-2023/test/test_gt.json'

# ── Training hyper-parameters ──────────────────────────────────────────────
ARCH              = 'mobilenet_v3_large'   # mobilenet_v3_small | mobilenet_v3_large | resnet18
EPOCHS            = 25
BATCH_SIZE        = 32
LR                = 1e-3
IMG_SIZE          = 128
MAX_PER_TRACKLET  = 25
FREEZE_EPOCHS     = 2
LABEL_SMOOTHING   = 0.1
PATIENCE          = 5
VAL_SPLIT         = 0.1
WORKERS           = 2
USE_KEYFRAMES     = True    # filter blurry frames during training

# ── Inference settings ─────────────────────────────────────────────────────
PREDICT_BATCH     = 64
USE_TTA           = True    # test-time augmentation
USE_KF_PREDICT    = True    # keyframe selection during inference

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Python: {sys.executable}')
print('Configuration ready.')

Working directory: /content/COSC419B_2025W2
Python: /usr/bin/python3
Configuration ready.


## 1. Preprocessing — torso crops
Runs MediaPipe pose detection on every image and saves tight torso crops.  
**Skip this cell if crops already exist.**  Re-running is safe (skips existing crops).

In [ ]:
cmd = [
    sys.executable, 'src/preprocess_crops.py',
    '--data-dir', DATA_DIR,
    '--split',    'all',
    '--workers',  str(os.cpu_count() or 4),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)
print('\nExit code:', result.returncode)

cmd = [
    sys.executable, 'src/train.py',
    '--data-dir',        DATA_DIR,
    '--crops-dir',       CROPS_DIR_TRAIN,
    '--arch',            ARCH,
    '--epochs',          str(EPOCHS),
    '--batch-size',      str(BATCH_SIZE),
    '--lr',              str(LR),
    '--img-size',        str(IMG_SIZE),
    '--max-per-tracklet',str(MAX_PER_TRACKLET),
    '--freeze-epochs',   str(FREEZE_EPOCHS),
    '--label-smoothing', str(LABEL_SMOOTHING),
    '--patience',        str(PATIENCE),
    '--val-split',       str(VAL_SPLIT),
    '--workers',         str(WORKERS),
    '--output-dir',      OUTPUT_DIR,
]
if USE_KEYFRAMES:
    cmd.append('--keyframes')

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)
print('\nExit code:', result.returncode)

In [10]:
cmd = [
    sys.executable, 'src/train.py',
    '--data-dir',        DATA_DIR,
    '--crops-dir',       CROPS_DIR_TRAIN,
    '--arch',            ARCH,
    '--epochs',          str(EPOCHS),
    '--batch-size',      str(BATCH_SIZE),
    '--lr',              str(LR),
    '--img-size',        str(IMG_SIZE),
    '--max-per-tracklet',str(MAX_PER_TRACKLET),
    '--freeze-epochs',   str(FREEZE_EPOCHS),
    '--label-smoothing', str(LABEL_SMOOTHING),
    '--patience',        str(PATIENCE),
    '--val-split',       str(VAL_SPLIT),
    '--workers',         str(WORKERS),
    '--output-dir',      OUTPUT_DIR,
]
if USE_KEYFRAMES:
    cmd.append('--keyframes')

print('Running:', ' '.join(cmd))
print('CWD:', os.getcwd())
result = subprocess.run(cmd, stderr=subprocess.STDOUT)
print('\nExit code:', result.returncode)

Running: /usr/bin/python3 src/train.py --data-dir data/jersey-2023 --crops-dir data/jersey-2023/train/crops --arch mobilenet_v3_large --epochs 25 --batch-size 32 --lr 0.001 --img-size 128 --max-per-tracklet 25 --freeze-epochs 2 --label-smoothing 0.1 --patience 5 --val-split 0.1 --workers 2 --output-dir outputs --keyframes
CWD: /content/COSC419B_2025W2

Exit code: 2


### Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = os.path.join(OUTPUT_DIR, 'history.json')
with open(history_path) as f:
    history = json.load(f)

epochs     = [h['epoch']      for h in history]
train_acc  = [h['train_acc']  for h in history]
val_acc    = [h['val_acc']    for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss']   for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_acc,  label='Train')
ax1.plot(epochs, val_acc,    label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)

ax2.plot(epochs, train_loss, label='Train')
ax2.plot(epochs, val_loss,   label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True)

plt.suptitle(f'Best val acc: {max(val_acc):.4f} (epoch {epochs[val_acc.index(max(val_acc))]})')
plt.tight_layout()
plt.show()

## 3. Prediction

In [ ]:
cmd = [
    sys.executable, 'src/predict.py',
    '--data-dir',   DATA_DIR,
    '--crops-dir',  CROPS_DIR_TEST,
    '--checkpoint', CHECKPOINT,
    '--output',     PREDICTIONS_FILE,
    '--img-size',   str(IMG_SIZE),
    '--batch-size', str(PREDICT_BATCH),
    '--gt',         GT_TEST,
]
if USE_TTA:
    cmd.append('--tta')
if USE_KF_PREDICT:
    cmd.append('--keyframes')

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)
print('\nExit code:', result.returncode)

## 4. Evaluation

In [ ]:
cmd = [
    sys.executable, 'src/run_evaluate.py',
    '--predictions',  PREDICTIONS_FILE,
    '--ground-truth', GT_TEST,
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)
print('\nExit code:', result.returncode)

### Prediction breakdown

In [ ]:
with open(PREDICTIONS_FILE) as f:
    predictions = json.load(f)
with open(GT_TEST) as f:
    gt = json.load(f)

total = correct = legible_total = legible_correct = illegible_total = illegible_correct = 0
for tid, gt_num in gt.items():
    if tid not in predictions:
        continue
    pred = predictions[tid]
    total   += 1
    correct += pred == gt_num
    if gt_num == -1:
        illegible_total   += 1
        illegible_correct += pred == -1
    else:
        legible_total   += 1
        legible_correct += pred == gt_num

print(f'Overall accuracy :  {correct}/{total} = {100*correct/total:.1f}%')
if legible_total:
    print(f'Legible accuracy :  {legible_correct}/{legible_total} = {100*legible_correct/legible_total:.1f}%')
if illegible_total:
    print(f'Illegible accuracy: {illegible_correct}/{illegible_total} = {100*illegible_correct/illegible_total:.1f}%')

# Distribution of predicted numbers
from collections import Counter
pred_counts = Counter(predictions.values())
gt_counts   = Counter(gt.values())

numbers = sorted(set(gt_counts) | set(pred_counts))
gt_vals   = [gt_counts.get(n, 0)   for n in numbers]
pred_vals = [pred_counts.get(n, 0) for n in numbers]

fig, ax = plt.subplots(figsize=(16, 4))
x = range(len(numbers))
ax.bar([i - 0.2 for i in x], gt_vals,   width=0.4, label='Ground truth', alpha=0.7)
ax.bar([i + 0.2 for i in x], pred_vals, width=0.4, label='Predicted',    alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels([str(n) for n in numbers], rotation=90, fontsize=7)
ax.set_xlabel('Jersey number (-1 = illegible)')
ax.set_ylabel('Count')
ax.set_title('Prediction distribution vs ground truth')
ax.legend()
plt.tight_layout()
plt.show()